# 📊 Pocket OTC AI Analyzer
## واجهة Google Colab
أدخل Telegram Bot Token وOpenRouter API Key في حقول منفصلة. لا يتم حفظ المفاتيح في GitHub.

In [ ]:
!pip -q install -r https://raw.githubusercontent.com/mohmb142/Jjjjjjj/main/colab_requirements.txt

import os, subprocess, sys, asyncio, importlib
import ipywidgets as widgets
from IPython.display import display, HTML

telegram_box = widgets.Password(description='Telegram:', placeholder='Bot Token', layout=widgets.Layout(width='100%'), style={'description_width':'90px'})
openrouter_box = widgets.Password(description='OpenRouter:', placeholder='API Key', layout=widgets.Layout(width='100%'), style={'description_width':'90px'})
model_box = widgets.Text(value='google/gemini-2.5-flash', description='Vision:', layout=widgets.Layout(width='100%'), style={'description_width':'90px'})
status = widgets.Output(layout=widgets.Layout(border='1px solid #ccc', padding='10px', margin='10px 0'))
test_btn = widgets.Button(description='🧪 اختبار OpenRouter', button_style='info', layout=widgets.Layout(width='100%'))
start_btn = widgets.Button(description='🚀 تشغيل البوت', button_style='success', layout=widgets.Layout(width='100%'))

def card(title, widget):
    return widgets.VBox([widgets.HTML(f'<h4 style=\"margin:0 0 8px 0\">{title}</h4>'), widget], layout=widgets.Layout(border='1px solid #ccc', padding='14px', margin='8px 0', width='100%'))

display(HTML('<h2>🤖 Pocket OTC AI Analyzer</h2><p>📷 أرسل صورة الشارت إلى Telegram ليتم تحليلها عبر OpenRouter Vision.</p><p>⚠️ تحليل فقط — لا يتم تنفيذ أي صفقة.</p>'))
display(card('🔐 Telegram Bot Token', telegram_box))
display(card('🧠 OpenRouter API Key', openrouter_box))
display(card('👁️ Vision Model', model_box))
display(test_btn, start_btn, status)

def set_env():
    token = telegram_box.value.strip()
    key = openrouter_box.value.strip()
    model = model_box.value.strip()
    if not token: raise ValueError('أدخل Telegram Bot Token')
    if not key: raise ValueError('أدخل OpenRouter API Key')
    if not model: raise ValueError('أدخل Vision Model')
    os.environ['TELEGRAM_BOT_TOKEN'] = token
    os.environ['OPENROUTER_API_KEY'] = key
    os.environ['OPENROUTER_MODEL'] = model

def on_test(_):
    with status:
        status.clear_output()
        try:
            set_env()
            import httpx
            r = httpx.get('https://openrouter.ai/api/v1/models', headers={'Authorization':'Bearer '+os.environ['OPENROUTER_API_KEY']}, timeout=20)
            r.raise_for_status()
            print('✅ OpenRouter متصل والمفتاح مقبول.')
        except Exception as e:
            print('❌ فشل الاختبار:', e)

def on_start(_):
    with status:
        status.clear_output()
        try:
            set_env()
            if not os.path.isdir('app'):
                subprocess.run(['git','clone','-q','https://github.com/mohmb142/Jjjjjjj.git','app'], check=True)
            else:
                subprocess.run(['git','-C','app','pull','-q'], check=True)
            os.chdir('app')
            sys.path.insert(0, os.getcwd())
            # Colab keeps imported modules in sys.modules. Reload the repository
            # module so an older telegram_bot.py cannot be executed accidentally.
            if 'telegram_bot' in sys.modules:
                telegram_bot = importlib.reload(sys.modules['telegram_bot'])
            else:
                import telegram_bot
            print('🚀 تشغيل Telegram Bot...')
            task = telegram_bot.run()
            if task is not None:
                print('✅ تم تشغيل Telegram polling داخل Colab. اترك جلسة Colab قيد التشغيل.')
        except Exception as e:
            print('❌ تعذر التشغيل:', e)

test_btn.on_click(on_test)
start_btn.on_click(on_start)
